# lib_prep — 공통 전처리 함수 (담당: core)

여러 노트북이 같이 쓰는 함수만 둡니다. 2026-11-24 `preprocess.py`의 코드를 그대로 옮겼고, 바뀐 곳은 두 군데입니다.

1. **원핫 범주 고정:** 빈도 0.5% 미만 범주를 회차마다 판단하지 않고, 01_prep이 Dev 전체로 한 번 정한 목록(`contract/categories.json`)을 씁니다. 목록 밖 범주는 `OTHER`로 바꿉니다(01_eda_audit의 변수 선별과 같은 규칙). 그래서 회차가 바뀌어도, Test에서도 열이 같습니다.
2. **열 확인:** `check_columns()`가 fit 결과를 `contract/columns_*.json`과 비교합니다.

- **Stage 1** (행마다 고정 규칙, 01_prep과 05_test가 같이 씀): `load_and_prepare`
- **Stage 2** (회차마다 Train에서 fit): `make_preprocessor(model_kind)`. 스케일러는 logit일 때만
- 저장한 전처리기·모형을 다시 불러오는 노트북은 **먼저 이 노트북을 실행**해야 합니다(직접 만든 변환기 클래스가 여기 있음).

In [ ]:
# 0. 불러오기와 버전 해시
if "NB_DIR" not in globals():                      # 이 노트북을 직접 열었을 때는 config부터
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
from typing import Literal
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def _lib_hash():
    """이 노트북 코드 칸의 해시. 고치면 바뀐다 (팀원이 옛 버전을 쓰는지 확인용)."""
    p = NB_DIR / "lib_prep.ipynb"
    cells = json.loads(p.read_text(encoding="utf-8"))["cells"]
    src = "\n".join("".join(c["source"]) for c in cells if c["cell_type"] == "code")
    return hashlib.sha256(src.encode()).hexdigest()[:12]

LIB_PREP_HASH = _lib_hash()
print(f"[lib_prep] 해시 {LIB_PREP_HASH}")

In [ ]:
# 1. 변수표와 금지 칼럼 (preprocess.py 2026-11-24 그대로)
FEATURE_SPEC: dict[str, tuple[str, str, str, int, str]] = {
    # --- Core 5Cs (선행문헌 표준 16개 원천 변수) ---
    "fico_range_low":             ("fico_range_low",                 "raw",       "median",      -1, "core"),
    "earliest_cr_line":           ("credit_hist_len",                "cr_hist",   "median",      -1, "core"),
    "delinq_2yrs":                ("delinq_2yrs",                    "raw",       "median",      +1, "core"),
    "inq_last_6mths":             ("inq_last_6mths",                 "raw",       "median",      +1, "core"),
    "pub_rec":                    ("pub_rec",                        "raw",       "median",      +1, "core"),
    "open_acc":                   ("open_acc",                       "raw",       "median",       0, "core"),
    "total_acc":                  ("total_acc",                      "raw",       "median",       0, "core"),
    "annual_inc":                 ("log_annual_inc",                 "log1p_pos", "median",      -1, "core"),
    "emp_length":                 ("emp_length",                     "emp_len",   "median_flag",  0, "core"),  # 2026-11-26: -1→0 (0~9년 평평, PDP 약함)
    "dti":                        ("dti",                            "dti",       "median",      +1, "core"),
    "revol_util":                 ("revol_util",                     "pct",       "median",      +1, "core"),
    "revol_bal":                  ("log_revol_bal",                  "log1p",     "median",       0, "core"),
    "loan_amnt":                  ("log_loan_amnt",                  "log1p",     "median",      +1, "core"),
    "term":                       ("term_60m",                       "term60",    "median",      +1, "core"),
    "home_ownership":             ("home_ownership",                 "cat",       "cat",          0, "core"),
    "purpose":                    ("purpose",                        "cat",       "cat",          0, "core"),
    # --- Extended t0 (6단계 감사 통과, 2013+ 전 기간 가용, |r|<=0.75, VIF<5.0) ---
    "acc_open_past_24mths":       ("acc_open_past_24mths",           "raw",       "median",      +1, "ext"),
    "collections_12_mths_ex_med": ("collections_12_mths_ex_med",     "raw",       "median",      +1, "ext"),
    "tot_coll_amt":               ("log_tot_coll_amt",               "log1p",     "median",       0, "ext"),
    "tot_cur_bal":                ("log_tot_cur_bal",                "log1p",     "median",       0, "ext"),
    "total_bal_ex_mort":          ("log_total_bal_ex_mort",          "log1p",     "median",       0, "ext"),
    "total_bc_limit":             ("log_total_bc_limit",             "log1p",     "median",      -1, "ext"),
    "total_il_high_credit_limit": ("log_total_il_high_credit_limit", "log1p",     "median",       0, "ext"),
    "mo_sin_rcnt_rev_tl_op":      ("mo_sin_rcnt_rev_tl_op",          "raw",       "median",       0, "ext"),
    "mo_sin_rcnt_tl":             ("mo_sin_rcnt_tl",                 "raw",       "median",      -1, "ext"),
    "mort_acc":                   ("mort_acc",                       "raw",       "median",      -1, "ext"),
    "num_accts_ever_120_pd":      ("num_accts_ever_120_pd",          "raw",       "median",      +1, "ext"),
    "num_actv_bc_tl":             ("num_actv_bc_tl",                 "raw",       "median",       0, "ext"),
    "num_bc_tl":                  ("num_bc_tl",                      "raw",       "median",       0, "ext"),
    "num_tl_90g_dpd_24m":         ("num_tl_90g_dpd_24m",             "raw",       "median",       0, "ext"),
    "pub_rec_bankruptcies":       ("pub_rec_bankruptcies",           "raw",       "median",       0, "ext"),
    "pct_tl_nvr_dlq":             ("pct_tl_nvr_dlq",                 "raw",       "median",       0, "ext"),
    "mths_since_recent_bc":       ("mths_since_recent_bc",           "raw",       "median",      -1, "ext"),
    "bc_open_to_buy":             ("log_bc_open_to_buy",             "log1p",     "median",      -1, "ext"),
    "percent_bc_gt_75":           ("percent_bc_gt_75",               "raw",       "median",      +1, "ext"),
    "mo_sin_old_il_acct":         ("mo_sin_old_il_acct",             "raw",       "median",       0, "ext"),
    "mths_since_recent_inq":      ("mths_since_recent_inq",          "raw",       "max_flag",    -1, "ext"),
}

FORBIDDEN_COLS: set[str] = {
    "id", "member_id", "url", "policy_code", "desc", "title", "emp_title",
    "grade", "sub_grade", "int_rate", "installment", "verification_status", "verification_status_joint", "eff_pti",
    "funded_amnt", "funded_amnt_inv", "initial_list_status", "issue_d",
    "addr_state", "zip_code",
    "loan_status", "pymnt_plan", "out_prncp", "out_prncp_inv",
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low", "debt_settlement_flag",
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status",
    "deferral_term", "hardship_amount", "hardship_start_date", "hardship_end_date",
    "payment_plan_start_date", "hardship_length", "hardship_dpd", "hardship_loan_status",
    "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
    "hardship_last_payment_amount", "y", "rf_ann", "mirr", "excess_ret", "hpr_ann", "fv_total",
}

META_INPUT_COLS = [
    "id", "issue_d", "term", "loan_amnt", "funded_amnt", "grade",
    "loan_status", "total_pymnt", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt",
]

CAT_REFERENCE = {"home_ownership": "MORTGAGE", "purpose": "DEBT_CONSOLIDATION"}

RESOLVED_PAID = {"Fully Paid"}

RESOLVED_DEFAULT = {"Charged Off", "Default"}


def get_feature_spec(mode: Literal["screened_t0", "core_t0"] = "screened_t0") -> dict[str, tuple[str, str, str, int, str]]:
    if mode == "core_t0":
        return {k: v for k, v in FEATURE_SPEC.items() if v[4] == "core"}
    return dict(FEATURE_SPEC)

In [ ]:
# 2. Stage 1: 표본 필터, 국채 매칭, MIRR, 행마다 파싱 (preprocess.py 그대로)
def parse_month(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series.astype("string").str.strip(), format="%b-%Y", errors="coerce")


def parse_term_months(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    return pd.to_numeric(series.astype("string").str.extract(r"(\d+)")[0], errors="coerce")


def months_between(later_dt: pd.Series, earlier_dt: pd.Series) -> pd.Series:
    return (later_dt.dt.year - earlier_dt.dt.year) * 12.0 + (later_dt.dt.month - earlier_dt.dt.month)


def drop_corrupted_rows(df: pd.DataFrame) -> pd.DataFrame:
    stray = df["id"].astype("string").str.strip().fillna("") == "Loans that do not meet the credit policy"
    empty = df["loan_amnt"].isna() & df["issue_d"].isna()
    return df.loc[~(stray | empty)].copy()


def filter_cohort(
    df: pd.DataFrame,
    start_year: int = 2013,
    maturity_cutoff_ym: str | None = "2020-04",
    require_resolved: bool = True,
) -> pd.DataFrame:
    """깨진 행 제거 -> 종결 대출 -> 시작 연도 -> 만기 도래(우측 중도절단 제거)."""
    out = drop_corrupted_rows(df)
    if require_resolved:
        out = out.loc[out["loan_status"].isin(RESOLVED_PAID | RESOLVED_DEFAULT)]
    issue_dt = parse_month(out["issue_d"])
    term_m = parse_term_months(out["term"])
    mask = issue_dt.dt.year >= start_year
    if maturity_cutoff_ym is not None:
        cy, cm = map(int, maturity_cutoff_ym.split("-"))
        mask &= (issue_dt.dt.year * 12 + issue_dt.dt.month + term_m) <= (cy * 12 + cm)
    return out.loc[mask].reset_index(drop=True)


def load_treasury(treasury_path: str | Path) -> pd.DataFrame:
    """FRED 기반 월별 3Y/5Y 미국 국채 수익률 로드 (% -> 소수)."""
    tsy = pd.read_csv(treasury_path, usecols=["issue_d", "3y", "5y"])
    tsy["issue_d"] = tsy["issue_d"].astype("string").str.strip()
    tsy["rf_3y"] = pd.to_numeric(tsy["3y"], errors="coerce") / 100.0
    tsy["rf_5y"] = pd.to_numeric(tsy["5y"], errors="coerce") / 100.0
    return tsy.drop_duplicates("issue_d", keep="last")[["issue_d", "rf_3y", "rf_5y"]]


def compute_cashflow_mirr(
    funded_amnt: np.ndarray,
    term_m: np.ndarray,
    total_pymnt: np.ndarray,
    recoveries: np.ndarray,
    collection_recovery_fee: np.ndarray,
    last_pymnt_amnt: np.ndarray,
    k_months: np.ndarray,
    rf_ann: np.ndarray,
    recovery_lag: float = 5.0,
) -> dict[str, np.ndarray]:
    """
    월별 현금흐름 복원 폐형식(Closed-Form) 연환산 MIRR 및 초과수익률(MIRR - rf_ann).
    - 정기 상환액: P_reg = max(total_pymnt - recoveries, 0)
      k>=2이면 최종 납입(last_pymnt_amnt, 조기상환 포함)을 t=k에, 잔여액을 t=1..k-1 균등 배분
    - 부도 후 순추심액: R_net = max(recoveries - collection_recovery_fee, 0) at t_rec = min(k+5, T)
    - 모든 유입을 월 실효 무위험률 r_m = (1 + rf)^(1/12) - 1 로 계약 만기 T까지 복리 재투자
    """
    funded = np.asarray(funded_amnt, dtype=np.float64)
    T = np.asarray(term_m, dtype=np.float64)
    tp = np.nan_to_num(np.asarray(total_pymnt, dtype=np.float64), nan=0.0)
    rec = np.nan_to_num(np.asarray(recoveries, dtype=np.float64), nan=0.0)
    fee = np.nan_to_num(np.asarray(collection_recovery_fee, dtype=np.float64), nan=0.0)
    last_amt = np.nan_to_num(np.asarray(last_pymnt_amnt, dtype=np.float64), nan=0.0)
    rf = np.asarray(rf_ann, dtype=np.float64)

    r_m = (1.0 + rf) ** (1.0 / 12.0) - 1.0
    p_reg = np.maximum(tp - rec, 0.0)
    r_net = np.maximum(rec - fee, 0.0)
    k = np.where(np.isnan(k_months), 0.0, np.maximum(np.asarray(k_months, dtype=np.float64), 0.0))

    m = np.maximum(k - 1.0, 0.0)
    c_final = np.where(k >= 2.0, np.clip(last_amt, 0.0, p_reg), p_reg)
    c_reg = np.where(m > 0.0, np.where(k >= 2.0, p_reg - c_final, 0.0) / np.where(m > 0.0, m, 1.0), 0.0)

    with np.errstate(divide="ignore", invalid="ignore"):
        annuity_fv = np.where(
            np.abs(r_m) > 1e-12,
            ((1.0 + r_m) ** (T - m)) * (((1.0 + r_m) ** m - 1.0) / r_m),
            m,
        )
    fv_total = np.maximum(
        np.where(m > 0.0, c_reg * annuity_fv, 0.0)
        + c_final * ((1.0 + r_m) ** (T - k))
        + r_net * ((1.0 + r_m) ** (T - np.minimum(k + recovery_lag, T))),
        0.0,
    )
    mirr = (fv_total / funded) ** (12.0 / T) - 1.0
    hpr_ann = (np.maximum(p_reg + r_net, 0.0) / funded) ** (12.0 / T) - 1.0
    return {"mirr": mirr, "excess_ret": mirr - rf, "hpr_ann": hpr_ann, "fv_total": fv_total}


def build_meta(df: pd.DataFrame, treasury_df: pd.DataFrame, is_labeled: bool = True) -> pd.DataFrame:
    """Dev / Test 공용 평가 전용 메타 테이블 생성 (예측 피처 X와 엄격 분리, RangeIndex 유지)."""
    issue_dt = parse_month(df["issue_d"])
    term_m = parse_term_months(df["term"])
    issue_key = df["issue_d"].astype("string").str.strip()
    rf_3y = issue_key.map(treasury_df.set_index("issue_d")["rf_3y"]).to_numpy(dtype=np.float64)
    rf_5y = issue_key.map(treasury_df.set_index("issue_d")["rf_5y"]).to_numpy(dtype=np.float64)
    rf_ann = np.where(term_m.to_numpy() == 36, rf_3y, np.where(term_m.to_numpy() == 60, rf_5y, np.nan))
    if np.isnan(rf_ann).any():
        raise ValueError(f"국채 수익률 매칭 실패 {int(np.isnan(rf_ann).sum())}건")

    meta = pd.DataFrame({
        "id": df["id"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).to_numpy(),
        "issue_d": issue_key.to_numpy(),
        "issue_year": issue_dt.dt.year.to_numpy(dtype=np.int32),
        "term_m": term_m.to_numpy(dtype=np.int32),
        "loan_amnt": pd.to_numeric(df["loan_amnt"], errors="coerce").to_numpy(dtype=np.float64),
        "rf_ann": rf_ann,
    })
    if "grade" in df.columns:
        meta["grade"] = df["grade"].astype("string").str.strip().to_numpy()

    if is_labeled and {"loan_status", "total_pymnt"}.issubset(df.columns):
        status = df["loan_status"].astype("string").str.strip()
        meta["y"] = np.where(status.isin(RESOLVED_DEFAULT), 1, np.where(status.isin(RESOLVED_PAID), 0, -1)).astype(np.int8)
        funded = (
            pd.to_numeric(df["funded_amnt"], errors="coerce").fillna(meta["loan_amnt"])
            if "funded_amnt" in df.columns else meta["loan_amnt"]
        ).to_numpy(dtype=np.float64)
        get_num = lambda c: pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=np.float64) if c in df.columns else np.zeros(len(df))
        last_dt = parse_month(df["last_pymnt_d"]) if "last_pymnt_d" in df.columns else pd.Series(pd.NaT, index=df.index)
        k_m = months_between(last_dt, issue_dt).to_numpy(dtype=np.float64)
        res = compute_cashflow_mirr(
            funded, term_m.to_numpy(dtype=np.float64), get_num("total_pymnt"),
            get_num("recoveries"), get_num("collection_recovery_fee"), get_num("last_pymnt_amnt"),
            k_m, rf_ann,
        )
        for k, v in res.items():
            meta[k] = v
    return meta


def parse_raw_features(df: pd.DataFrame, mode: Literal["screened_t0", "core_t0"] = "screened_t0") -> pd.DataFrame:
    """행 독립(Row-wise) 1회 정제 — Train 통계 미사용, RangeIndex(0..N-1) 유지."""
    spec = get_feature_spec(mode)
    issue_dt = parse_month(df["issue_d"])
    cols: dict[str, np.ndarray] = {}

    for raw_col, (out_name, tf, _, _, _) in spec.items():
        if tf == "cat":
            s = df[raw_col].astype("string").str.strip().str.upper()
            cols[out_name] = s.mask(s.isin(["", "NAN", "NONE", "<NA>"])).fillna("OTHER").to_numpy(dtype=object)
        elif tf == "term60":
            cols[out_name] = (parse_term_months(df[raw_col]) == 60).astype(np.float64).to_numpy()
        elif tf == "cr_hist":
            cols[out_name] = months_between(issue_dt, parse_month(df[raw_col])).clip(lower=0.0).to_numpy(dtype=np.float64)
        elif tf == "emp_len":
            s = df[raw_col].astype("string").str.strip().str.lower()
            v = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce").astype(np.float64)
            cols[out_name] = v.mask(s.str.startswith("<", na=False), 0.0).clip(0.0, 10.0).to_numpy()
        elif tf == "pct":
            s = df[raw_col].astype("string").str.strip().str.rstrip("%")
            cols[out_name] = pd.to_numeric(s, errors="coerce").clip(0.0, 150.0).to_numpy(dtype=np.float64)
        elif tf == "dti":
            v = pd.to_numeric(df[raw_col], errors="coerce")
            cols[out_name] = v.where((v >= 0.0) & (v < 999.0), np.nan).to_numpy(dtype=np.float64)
        elif tf == "log1p_pos":
            v = pd.to_numeric(df[raw_col], errors="coerce")
            cols[out_name] = np.log1p(v.where(v > 0.0, np.nan)).to_numpy(dtype=np.float64)
        elif tf == "log1p":
            cols[out_name] = np.log1p(pd.to_numeric(df[raw_col], errors="coerce").clip(lower=0.0)).to_numpy(dtype=np.float64)
        else:  # raw
            cols[out_name] = pd.to_numeric(df[raw_col], errors="coerce").clip(lower=0.0).to_numpy(dtype=np.float64)

    return pd.DataFrame(cols)


def load_and_prepare(
    csv_path: str | Path,
    treasury_path: str | Path,
    mode: Literal["screened_t0", "core_t0"] = "screened_t0",
    start_year: int = 2013,
    maturity_cutoff_ym: str | None = "2020-04",
    is_labeled: bool = True,
    apply_cohort_filter: bool = True,
    chunksize: int = 250_000,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """[Stage 1 진입점] 필요한 칼럼만 1회 로드하여 (raw_X, meta) 반환 (Dev·Test 공용)."""
    spec = get_feature_spec(mode)
    wanted = set(spec) | set(META_INPUT_COLS)
    # 2026-11-26 수정: 청크(25만 행) 단위로 읽고 거름. 필터가 행 단위라 한 번에 읽을 때와 결과가 같고 메모리는 약 1/3
    parts = []
    for ch in pd.read_csv(csv_path, usecols=lambda c: c in wanted, low_memory=False, chunksize=chunksize):
        parts.append(filter_cohort(ch, start_year, maturity_cutoff_ym, require_resolved=is_labeled)
                     if apply_cohort_filter else drop_corrupted_rows(ch))
    df = pd.concat(parts, ignore_index=True)
    del parts
    treasury_df = load_treasury(treasury_path)
    return parse_raw_features(df, mode=mode), build_meta(df, treasury_df, is_labeled=is_labeled)

In [ ]:
# 3. Stage 2 변환기 (preprocess.py 그대로)
class _QuantileClipper(BaseEstimator, TransformerMixin):
    """Train 분위수 기반 상위 윈저라이징 (결측은 유지 후 다음 Imputer로 전달)."""
    def __init__(self, q: float = 0.995):
        self.q = q

    def fit(self, X, y=None):
        self.cols_ = list(X.columns) if hasattr(X, "columns") else [f"x{i}" for i in range(X.shape[1])]
        self.upper_ = np.nanquantile(np.asarray(X, dtype=np.float64), self.q, axis=0)
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=np.float64)
        return np.where(np.isnan(arr), np.nan, np.clip(arr, 0.0, self.upper_))

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features if input_features is not None else self.cols_)


class _MaxImputer(BaseEstimator, TransformerMixin):
    """구조적 결측('최근 조회 이력 없음') -> Train 최댓값 대체."""
    def fit(self, X, y=None):
        self.cols_ = list(X.columns) if hasattr(X, "columns") else [f"x{i}" for i in range(X.shape[1])]
        self.max_ = np.nanmax(np.asarray(X, dtype=np.float64), axis=0)
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=np.float64)
        return np.where(np.isnan(arr), self.max_, arr)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features if input_features is not None else self.cols_)


class _MissingFlag(BaseEstimator, TransformerMixin):
    """결측 여부 0/1 지시변수 생성 (스케일링 대상에서 자동 제외)."""
    def fit(self, X, y=None):
        self.cols_ = list(X.columns) if hasattr(X, "columns") else [f"x{i}" for i in range(X.shape[1])]
        return self

    def transform(self, X):
        return np.isnan(np.asarray(X, dtype=np.float64)).astype(np.float64)

    def get_feature_names_out(self, input_features=None):
        names = list(input_features if input_features is not None else self.cols_)
        return np.asarray([f"{c}_na" for c in names])


class _RareToOther(BaseEstimator, TransformerMixin):
    """고정 목록(keep) 밖의 범주를 'OTHER'로. 목록은 01_prep이 Dev 전체로 한 번 정한다."""
    def __init__(self, keep=None):
        self.keep = keep

    def fit(self, X, y=None):
        self.cols_ = list(X.columns) if hasattr(X, "columns") else [f"x{i}" for i in range(X.shape[1])]
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.cols_).copy()
        for c in self.cols_:
            X[c] = X[c].where(X[c].isin(self.keep[c]), "OTHER")
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features if input_features is not None else self.cols_)

In [ ]:
# 4. Stage 2 조립: 회차마다 Train에서 fit
def rare_category_levels(raw_X, mode="screened_t0", min_freq=None):
    """Dev 전체에서 빈도 min_freq 이상인 범주 + 'OTHER'. y를 쓰지 않는다. 01_prep에서만 호출."""
    min_freq = PREP["rare_cat_min_freq"] if min_freq is None else min_freq
    spec = get_feature_spec(mode)
    out = {}
    for _, (name, tf, _, _, _) in spec.items():
        if tf == "cat":
            freq = raw_X[name].value_counts(normalize=True)
            out[name] = sorted(set(freq[freq >= min_freq].index) | {"OTHER"})
    return out


def load_categories():
    return json.loads((CONTRACT_DIR / "categories.json").read_text(encoding="utf-8"))


def make_preprocessor(model_kind, categories=None, mode="screened_t0"):
    """preprocess.build_preprocessor와 같은 구성. 다른 점: 범주는 고정 목록, 스케일러는 logit만."""
    scale = (model_kind == "logit")
    categories = load_categories() if categories is None else categories
    spec = get_feature_spec(mode)
    dti_cols = [out for _, (out, tf, _, _, _) in spec.items() if tf == "dti"]
    bin_cols = [out for _, (out, tf, _, _, _) in spec.items() if tf == "term60"]
    max_flag_cols = [out for _, (out, _, imp, _, _) in spec.items() if imp == "max_flag"]
    med_flag_cols = [out for _, (out, _, imp, _, _) in spec.items() if imp == "median_flag"]
    cat_cols = [out for _, (out, tf, _, _, _) in spec.items() if tf == "cat"]
    med_num_cols = [out for _, (out, tf, imp, _, _) in spec.items() if imp == "median" and tf not in ("dti", "term60")]

    sc = [("scale", StandardScaler())] if scale else []
    transformers = [
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), *sc]), med_num_cols),
        ("dti", Pipeline([("clip", _QuantileClipper(PREP["dti_clip_q"])), ("imp", SimpleImputer(strategy="median")), *sc]), dti_cols),
        ("bin", SimpleImputer(strategy="median"), bin_cols),
    ]
    if med_flag_cols:
        transformers += [("med_num", Pipeline([("imp", SimpleImputer(strategy="median")), *sc]), med_flag_cols),
                         ("med_na", _MissingFlag(), med_flag_cols)]
    if max_flag_cols:
        transformers += [("max_num", Pipeline([("imp", _MaxImputer()), *sc]), max_flag_cols),
                         ("max_na", _MissingFlag(), max_flag_cols)]
    if cat_cols:
        ohe = OneHotEncoder(categories=[categories[c] for c in cat_cols], drop=[CAT_REFERENCE[c] for c in cat_cols],
                            handle_unknown="error", sparse_output=False)
        transformers.append(("cat", Pipeline([("rare", _RareToOther(keep=categories)), ("ohe", ohe)]), cat_cols))
    return ColumnTransformer(transformers, verbose_feature_names_out=False).set_output(transform="pandas")


def mono_constraints(fitted_ct: ColumnTransformer) -> tuple[int, ...]:
    """학습된 ColumnTransformer 출력 칼럼 순서에 100% 동기화된 XGBoost 단조성 튜플 반환 + 누수 검사."""
    out_cols = list(fitted_ct.get_feature_names_out())
    leaked = FORBIDDEN_COLS & set(out_cols)
    if leaked:
        raise ValueError(f"[누수 차단] 금지 칼럼 감지: {sorted(leaked)}")
    mono_map = {out_name: sign for _, (out_name, _, _, sign, _) in FEATURE_SPEC.items()}
    return tuple(mono_map.get(c, 0) for c in out_cols)


def check_columns(fitted_ct, model_kind):
    """fit된 전처리기의 출력 열이 contract/columns_*.json과 같은지. 다르면 멈춘다."""
    name = "columns_logit.json" if model_kind == "logit" else "columns_xgb.json"
    expected = json.loads((CONTRACT_DIR / name).read_text(encoding="utf-8"))
    got = list(fitted_ct.get_feature_names_out())
    assert got == expected, f"전처리 출력 열이 약속과 다름: 빠짐 {set(expected) - set(got)}, 추가 {set(got) - set(expected)}"
    return got


def compute_vif(X_df: pd.DataFrame) -> pd.Series:
    """상관행렬 역행렬 diag(R^{-1}) 기반 정확 VIF (OLS 1/(1-R_j^2)와 수학적 동치)."""
    R = np.corrcoef(X_df.to_numpy(dtype=np.float64), rowvar=False)
    return pd.Series(np.diag(np.linalg.inv(R)), index=X_df.columns, name="VIF")

In [ ]:
# 5. 점검 (이 노트북을 직접 열었을 때만): MIRR 계산 단위 검증 (01_eda_audit 5번 칸)
if not globals().get("SKIP_LIB_TESTS", False):
    _u = compute_cashflow_mirr(
        funded_amnt=np.array([10000.0, 10000.0]), term_m=np.array([36.0, 36.0]),
        total_pymnt=np.array([0.0, 10000.0]), recoveries=np.array([0.0, 0.0]),
        collection_recovery_fee=np.array([0.0, 0.0]), last_pymnt_amnt=np.array([0.0, 10000.0]),
        k_months=np.array([np.nan, 0.0]), rf_ann=np.array([0.02, 0.03]))
    assert np.isclose(_u["mirr"][0], -1.0) and np.isclose(_u["excess_ret"][1], 0.0)
    print("[PASS] MIRR 단위 검증 (전액손실 = -100%, 0개월 원금상환 초과수익률 = 0%)")